# 데이터 불러오기

In [4]:
import pandas as pd

df = pd.read_csv('./data/all-data.csv',
                 header=None,
                 names=['label', 'sentence'],
                 encoding='latin-1')

print(df.shape)
print(df['label'].value_counts())
print(df.head())

(4846, 2)
label
neutral     2879
positive    1363
negative     604
Name: count, dtype: int64
      label                                           sentence
0   neutral  According to Gran , the company has no plans t...
1   neutral  Technopolis plans to develop in stages an area...
2  negative  The international electronic industry company ...
3  positive  With the new production plant the company woul...
4  positive  According to the company 's updated strategy f...


In [5]:
# 결측값 확인
print(df.isnull().sum())

# 문장 길이 분포 확인
df['length'] = df['sentence'].apply(len)
print(df['length'].describe())

label       0
sentence    0
dtype: int64
count    4846.000000
mean      128.132068
std        56.526180
min         9.000000
25%        84.000000
50%       119.000000
75%       163.000000
max       315.000000
Name: length, dtype: float64


In [6]:
from sklearn.model_selection import train_test_split

X = df['sentence']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(y_train.value_counts())


Train: 3876, Test: 970
label
neutral     2303
positive    1090
negative     483
Name: count, dtype: int64


# 토크나이징 작업

In [7]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 토크나이저 설정
MAX_WORDS = 5000   # 최대 단어 수
MAX_LEN = 100      # 문장 최대 길이

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

# 텍스트 → 숫자 시퀀스 변환
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

# 길이 통일 (패딩)
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding='post', truncating='post')

print(X_train_pad.shape)
print(X_test_pad.shape)
print(X_train_pad[0])  # 첫 번째 문장 숫자 변환 확인


I0000 00:00:1775022253.776746   55641 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1775022253.927113   55641 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775022255.678817   55641 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


(3876, 100)
(970, 100)
[   2  332 2431  113  123  808  577   26   20   49   39   96  418  148
   33    6 1778 1396   28 1779   14 2432 2024   28 1779    4  296    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0]


# 라벨 negative, neutral, positive 원핫인코딩

In [10]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

le = LabelEncoder()
y_train_enc = to_categorical(le.fit_transform(y_train), num_classes=3)
y_test_enc  = to_categorical(le.transform(y_test), num_classes=3)

print(le.classes_)       # 라벨 순서 확인
print(y_train_enc[0], y_train_enc[1], y_train_enc[2])    # 첫 번째 라벨 확인


['negative' 'neutral' 'positive']
[0. 0. 1.] [0. 1. 0.] [0. 0. 1.]


# MLP 모델

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Flatten, Dropout

VOCAB_SIZE = 5000
EMBED_DIM = 32

mlp_model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),  # 단어 → 벡터
    Flatten(),                                                 # 벡터 → 1차원
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')                            # 3개 클래스 출력
])

mlp_model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

mlp_model.summary()


/home/ubuntu/miniforge3/envs/dl_env/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1775022822.183912   55641 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5563 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history_mlp = mlp_model.fit(
    X_train_pad, y_train_enc,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/10


I0000 00:00:1775022873.062278   60057 service.cc:153] XLA service 0x7357cc031e70 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775022873.062308   60057 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9 (Driver: 12.7.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.20.0)
I0000 00:00:1775022873.097047   60057 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1775022873.248123   60057 cuda_dnn.cc:461] Loaded cuDNN version 92000
I0000 00:00:1775022873.283405   60057 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1794__.20


41/97 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5650 - loss: 0.9868

I0000 00:00:1775022875.619403   60057 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


80/97 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5784 - loss: 0.9640

I0000 00:00:1775022876.114400   60055 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1794__.20


97/97 ━━━━━━━━━━━━━━━━━━━━ 7s 36ms/step - accuracy: 0.5887 - loss: 0.9256 - val_accuracy: 0.6082 - val_loss: 0.8389
Epoch 2/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6632 - loss: 0.7652 - val_accuracy: 0.6933 - val_loss: 0.7174
Epoch 3/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8074 - loss: 0.4817 - val_accuracy: 0.6843 - val_loss: 0.7114
Epoch 4/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9355 - loss: 0.2026 - val_accuracy: 0.7165 - val_loss: 0.7874
Epoch 5/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9842 - loss: 0.0625 - val_accuracy: 0.7191 - val_loss: 0.8648
Epoch 6/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9958 - loss: 0.0237 - val_accuracy: 0.7088 - val_loss: 1.0162
Epoch 7/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9981 - loss: 0.0095 - val_accuracy: 0.7268 - val_loss: 1.1070
Epoch 8/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9990 - loss: 0.0086 - val_accuracy: 0.7126 - val_loss: 1.0961
Ep

In [13]:
# 테스트셋 최종 평가
mlp_loss, mlp_acc = mlp_model.evaluate(X_test_pad, y_test_enc, verbose=0)

# 결과 저장용 딕셔너리
results = {}
results['MLP'] = {
    'test_accuracy': mlp_acc,
    'test_loss': mlp_loss,
    'history': history_mlp
}

print(f"MLP 테스트 정확도: {mlp_acc:.4f}")
print(f"MLP 테스트 손실:   {mlp_loss:.4f}")


MLP 테스트 정확도: 0.7227
MLP 테스트 손실:   1.2607


- MLP 원본          → val_accuracy: 72.7% (과적합)
- MLP + EarlyStopping→ ?

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',      # 검증 손실 기준
    patience=3,              # 3번 연속 안 좋아지면 종료
    restore_best_weights=True # 가장 좋았던 시점으로 복원
)

mlp_model2 = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

mlp_model2.compile(optimizer='adam',
                   loss='categorical_crossentropy',
                   metrics=['accuracy'])

history_mlp2 = mlp_model2.fit(
    X_train_pad, y_train_enc,
    epochs=50,              # 넉넉히 설정, EarlyStopping이 알아서 멈춤
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/50


I0000 00:00:1775023517.887189   60056 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_17709__.20


92/97 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5744 - loss: 0.9486

I0000 00:00:1775023519.347874   60055 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_17709__.20


97/97 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.5916 - loss: 0.9166 - val_accuracy: 0.6289 - val_loss: 0.8444
Epoch 2/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6752 - loss: 0.7452 - val_accuracy: 0.6804 - val_loss: 0.7411
Epoch 3/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8181 - loss: 0.4403 - val_accuracy: 0.6907 - val_loss: 0.7672
Epoch 4/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9158 - loss: 0.2229 - val_accuracy: 0.7036 - val_loss: 0.8072
Epoch 5/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9845 - loss: 0.0676 - val_accuracy: 0.7384 - val_loss: 0.9027


In [18]:
mlp_loss, mlp_acc = mlp_model2.evaluate(X_test_pad, y_test_enc, verbose=0)

results = {}
results['MLP'] = {
    'test_accuracy': round(mlp_acc, 4),
    'test_loss': round(mlp_loss, 4)
}

print(f"MLP | 정확도: {mlp_acc:.4f} | 손실: {mlp_loss:.4f}")

MLP | 정확도: 0.6835 | 손실: 0.7393


- MLP는 Epoch 2에서 이미 한계에 도달함.
- 그 이후로는 훈련 데이터만 외우고(과적합) 새로운 데이터엔 오히려 성능이 떨어짐.

# CNN 모델

In [24]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop_cnn = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

cnn_model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),
    Conv1D(64, kernel_size=3, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])

cnn_model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

history_cnn = cnn_model.fit(
    X_train_pad, y_train_enc,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop_cnn],
    verbose=1
)


Epoch 1/50


I0000 00:00:1775024002.112364   60051 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_38180__.22


89/97 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5635 - loss: 0.9953

I0000 00:00:1775024003.888607   60056 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_38180__.22


97/97 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.5845 - loss: 0.9460 - val_accuracy: 0.6070 - val_loss: 0.8790
Epoch 2/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6526 - loss: 0.8134 - val_accuracy: 0.6881 - val_loss: 0.7400
Epoch 3/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7661 - loss: 0.5857 - val_accuracy: 0.7268 - val_loss: 0.6446
Epoch 4/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8758 - loss: 0.3661 - val_accuracy: 0.7474 - val_loss: 0.6325
Epoch 5/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9400 - loss: 0.1958 - val_accuracy: 0.7526 - val_loss: 0.6628
Epoch 6/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9755 - loss: 0.1005 - val_accuracy: 0.7500 - val_loss: 0.7542
Epoch 7/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9900 - loss: 0.0493 - val_accuracy: 0.7500 - val_loss: 0.8428


In [25]:
cnn_loss, cnn_acc = cnn_model.evaluate(X_test_pad, y_test_enc, verbose=0)

results['CNN'] = {
    'test_accuracy': round(cnn_acc, 4),
    'test_loss': round(cnn_loss, 4)
}

print(f"MLP | 정확도: {results['MLP']['test_accuracy']} | 손실: {results['MLP']['test_loss']}")
print(f"CNN | 정확도: {results['CNN']['test_accuracy']} | 손실: {results['CNN']['test_loss']}")



MLP | 정확도: 0.6835 | 손실: 0.7393
CNN | 정확도: 0.7309 | 손실: 0.7194


# GRU

In [30]:
early_stop_gru = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

gru_model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),
    GRU(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

gru_model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

history_gru = gru_model.fit(
    X_train_pad, y_train_enc,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop_gru],
    verbose=1
)


Epoch 1/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.5884 - loss: 0.9512 - val_accuracy: 0.6070 - val_loss: 0.9205
Epoch 2/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.5910 - loss: 0.9358 - val_accuracy: 0.6070 - val_loss: 0.9171
Epoch 3/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.5910 - loss: 0.9312 - val_accuracy: 0.6070 - val_loss: 0.9238
Epoch 4/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.5910 - loss: 0.9314 - val_accuracy: 0.6070 - val_loss: 0.9220
Epoch 5/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.5910 - loss: 0.9326 - val_accuracy: 0.6070 - val_loss: 0.9197


In [31]:
gru_loss, gru_acc = gru_model.evaluate(X_test_pad, y_test_enc, verbose=0)

results['GRU'] = {
    'test_accuracy': round(gru_acc, 4),
    'test_loss': round(gru_loss, 4)
}

print(f"MLP | 정확도: {results['MLP']['test_accuracy']} | 손실: {results['MLP']['test_loss']}")
print(f"CNN | 정확도: {results['CNN']['test_accuracy']} | 손실: {results['CNN']['test_loss']}")
print(f"GRU | 정확도: {results['GRU']['test_accuracy']} | 손실: {results['GRU']['test_loss']}")


MLP | 정확도: 0.6835 | 손실: 0.7393
CNN | 정확도: 0.7309 | 손실: 0.7194
GRU | 정확도: 0.5938 | 손실: 0.9261


# LSTM 

In [32]:
from tensorflow.keras.layers import LSTM

early_stop_lstm = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

lstm_model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

lstm_model.compile(optimizer='adam',
                   loss='categorical_crossentropy',
                   metrics=['accuracy'])

history_lstm = lstm_model.fit(
    X_train_pad, y_train_enc,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop_lstm],
    verbose=1
)


Epoch 1/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.5906 - loss: 0.9536 - val_accuracy: 0.6070 - val_loss: 0.9186
Epoch 2/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.5910 - loss: 0.9304 - val_accuracy: 0.6070 - val_loss: 0.9209
Epoch 3/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.5910 - loss: 0.9326 - val_accuracy: 0.6070 - val_loss: 0.9177
Epoch 4/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.5910 - loss: 0.9307 - val_accuracy: 0.6070 - val_loss: 0.9211
Epoch 5/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.5910 - loss: 0.9327 - val_accuracy: 0.6070 - val_loss: 0.9213
Epoch 6/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.5910 - loss: 0.9328 - val_accuracy: 0.6070 - val_loss: 0.9269


RNN 64로 올려서 학습 다시

In [33]:
EMBED_DIM = 64  # 32 → 64

early_stop_lstm = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

lstm_model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

lstm_model.compile(optimizer='adam',
                   loss='categorical_crossentropy',
                   metrics=['accuracy'])

history_lstm = lstm_model.fit(
    X_train_pad, y_train_enc,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop_lstm],
    verbose=1
)


Epoch 1/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.5842 - loss: 0.9484 - val_accuracy: 0.6070 - val_loss: 0.9324
Epoch 2/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.5910 - loss: 0.9372 - val_accuracy: 0.6070 - val_loss: 0.9228
Epoch 3/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.5910 - loss: 0.9313 - val_accuracy: 0.6070 - val_loss: 0.9349
Epoch 4/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.5910 - loss: 0.9357 - val_accuracy: 0.6070 - val_loss: 0.9183
Epoch 5/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.5910 - loss: 0.9328 - val_accuracy: 0.6070 - val_loss: 0.9197
Epoch 6/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.5910 - loss: 0.9317 - val_accuracy: 0.6070 - val_loss: 0.9174
Epoch 7/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.5910 - loss: 0.9309 - val_accuracy: 0.6070 - val_loss: 0.9196
Epoch 8/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.5910 - loss: 0.9306 - val_accuracy: 0.6070 - v

Input_length 제거

In [34]:
early_stop_lstm = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

lstm_model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM),   # input_length 제거
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

lstm_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),  # lr 높임
                   loss='categorical_crossentropy',
                   metrics=['accuracy'])

history_lstm = lstm_model.fit(
    X_train_pad, y_train_enc,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop_lstm],
    verbose=1
)


Epoch 1/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.5842 - loss: 0.9466 - val_accuracy: 0.6070 - val_loss: 0.9219
Epoch 2/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.5910 - loss: 0.9320 - val_accuracy: 0.6070 - val_loss: 0.9277
Epoch 3/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.5910 - loss: 0.9325 - val_accuracy: 0.6070 - val_loss: 0.9173
Epoch 4/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.5910 - loss: 0.9302 - val_accuracy: 0.6070 - val_loss: 0.9320
Epoch 5/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.5910 - loss: 0.9340 - val_accuracy: 0.6070 - val_loss: 0.9172
Epoch 6/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.5910 - loss: 0.9314 - val_accuracy: 0.6070 - val_loss: 0.9176
Epoch 7/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.5910 - loss: 0.9308 - val_accuracy: 0.6070 - val_loss: 0.9224
Epoch 8/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.5910 - loss: 0.9304 - val_accuracy: 0.6070 - v

# MLP, CNN, GRU, LSTM 비교

In [35]:
lstm_loss, lstm_acc = lstm_model.evaluate(X_test_pad, y_test_enc, verbose=0)

results['LSTM'] = {
    'test_accuracy': round(lstm_acc, 4),
    'test_loss': round(lstm_loss, 4),
    'note': '학습 실패 - 환경 이슈 추정'
}

print("===== 최종 결과 비교 =====")
print(f"MLP  | 정확도: {results['MLP']['test_accuracy']} | 손실: {results['MLP']['test_loss']}")
print(f"CNN  | 정확도: {results['CNN']['test_accuracy']} | 손실: {results['CNN']['test_loss']}")
print(f"GRU  | 정확도: {results['GRU']['test_accuracy']} | 손실: {results['GRU']['test_loss']} | 학습 실패")
print(f"LSTM | 정확도: {results['LSTM']['test_accuracy']} | 손실: {results['LSTM']['test_loss']} | 학습 실패")
print(f"\n최고 성능: CNN ({results['CNN']['test_accuracy']})")


===== 최종 결과 비교 =====
MLP  | 정확도: 0.6835 | 손실: 0.7393
CNN  | 정확도: 0.7309 | 손실: 0.7194
GRU  | 정확도: 0.5938 | 손실: 0.9261 | 학습 실패
LSTM | 정확도: 0.5938 | 손실: 0.9262 | 학습 실패

최고 성능: CNN (0.7309)


# 금융뉴스로 성능 테스트

In [61]:
from newspaper import Article

urls = [
    "https://finance.yahoo.com/sectors/energy/articles/no-other-choice-war-hit-031518842.html",
    "https://finance.yahoo.com/news/live/stock-market-today-dow-soars-1000-points-sp-500-and-nasdaq-surge-in-upbeat-end-to-brutal-quarter-184154822.html",
    "https://finance.yahoo.com/news/oil-plunges-stocks-soar-after-iranian-president-offers-first-signs-of-willingness-to-end-war-with-us-183158792.html",
    "https://finance.yahoo.com/news/airlines-face-price-hikes-lower-margins-as-iran-war-pressures-business-171745222.html"
]

full_texts = []
titles = []
for url in urls:
    try:
        article = Article(url)
        article.download()
        article.parse()
        full_texts.append(article.text)
        titles.append(article.title)
        print(f"완료: {article.title}")
    except Exception as e:
        print(f"실패: {url}")


완료: ‘There’s No Other Choice’: War-Hit Asian Buyers Grab Russian Oil
완료: Stock market today: Dow soars 1,000 points, S&P 500 and Nasdaq surge in upbeat end to brutal quarter
완료: Oil plunges, stocks soar after Iranian president offers first signs of willingness to end war with US
완료: Airlines face price hikes, lower margins as Iran war pressures business


## 본문 전체로 예측

In [62]:
from newspaper import Article
import numpy as np

urls = [
    "https://finance.yahoo.com/sectors/energy/articles/no-other-choice-war-hit-031518842.html",
    "https://finance.yahoo.com/news/live/stock-market-today-dow-soars-1000-points-sp-500-and-nasdaq-surge-in-upbeat-end-to-brutal-quarter-184154822.html",
    "https://finance.yahoo.com/news/oil-plunges-stocks-soar-after-iranian-president-offers-first-signs-of-willingness-to-end-war-with-us-183158792.html",
    "https://finance.yahoo.com/news/airlines-face-price-hikes-lower-margins-as-iran-war-pressures-business-171745222.html"
]

full_texts = []
for url in urls:
    try:
        article = Article(url)
        article.download()
        article.parse()
        full_texts.append(article.text)
    except:
        print(f"실패: {url}")

predict_sentiment(full_texts)


문장: (Bloomberg) — Energy-starved Asian nations are taking advantage of US sanction waivers to buy Russian oil to fill gaps caused by the Iran war.

The Philippines took its first cargo of ESPO crude in nearly six years, while South Korea’s first Russian naphtha shipment this year has arrived at Daesan port and is awaiting discharge, according to ship-tracking data. Other countries including Sri Lanka are in talks with Moscow over shipments.

The war in the Middle East between the US, Israel and Iran has pitched Asia into a severe energy crunch, with the near-total closure of the Strait of Hormuz choking off supplies. The disruption has left the region’s refiners desperate to secure alternative cargoes of oil and products.

“There’s no other choice,” said June Goh, an analyst at Sparta Commodities. “Refineries that do not have much flexibility will be the first to look for Russian crude, as they are relatively easy replacements for Middle Eastern supplies.”

Russia has emerged as one of

In [63]:
def predict_sentiment_pretty(texts, titles=None):
    for i, text in enumerate(texts):
        seq = tokenizer.texts_to_sequences([text])
        pad = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
        pred = cnn_model.predict(pad, verbose=0)[0]
        labels = le.classes_

        result = labels[np.argmax(pred)]
        emoji = {"positive": "🟢", "neutral": "🟡", "negative": "🔴"}

        print(f"{'='*60}")
        if titles:
            print(f"제목: {titles[i]}")
        print(f"예측: {emoji[result]} {result.upper()}")
        print(f"  negative: {pred[0]:.2%}")
        print(f"  neutral:  {pred[1]:.2%}")
        print(f"  positive: {pred[2]:.2%}")
        print()

predict_sentiment_pretty(full_texts, titles=titles)

제목: ‘There’s No Other Choice’: War-Hit Asian Buyers Grab Russian Oil
예측: 🟡 NEUTRAL
  negative: 3.46%
  neutral:  89.45%
  positive: 7.09%

제목: Stock market today: Dow soars 1,000 points, S&P 500 and Nasdaq surge in upbeat end to brutal quarter
예측: 🟢 POSITIVE
  negative: 2.37%
  neutral:  23.56%
  positive: 74.06%

제목: Oil plunges, stocks soar after Iranian president offers first signs of willingness to end war with US
예측: 🟡 NEUTRAL
  negative: 4.74%
  neutral:  89.30%
  positive: 5.96%

제목: Airlines face price hikes, lower margins as Iran war pressures business
예측: 🟡 NEUTRAL
  negative: 3.71%
  neutral:  56.18%
  positive: 40.11%



## 헤드라인 + 본문 첫 2~3문장

In [65]:
from nltk.tokenize import sent_tokenize

summary_texts = []
for text in full_texts:
    first_3 = " ".join(sent_tokenize(text)[:3])
    summary_texts.append(first_3)

predict_sentiment_pretty(summary_texts, titles=titles)
from nltk.tokenize import sent_tokenize

summary_texts = []
for text in full_texts:
    first_3 = " ".join(sent_tokenize(text)[:3])
    summary_texts.append(first_3)

predict_sentiment_pretty(summary_texts, titles=titles)

제목: ‘There’s No Other Choice’: War-Hit Asian Buyers Grab Russian Oil
예측: 🟡 NEUTRAL
  negative: 5.66%
  neutral:  86.38%
  positive: 7.95%

제목: Stock market today: Dow soars 1,000 points, S&P 500 and Nasdaq surge in upbeat end to brutal quarter
예측: 🟢 POSITIVE
  negative: 2.37%
  neutral:  23.56%
  positive: 74.06%

제목: Oil plunges, stocks soar after Iranian president offers first signs of willingness to end war with US
예측: 🟡 NEUTRAL
  negative: 4.74%
  neutral:  89.30%
  positive: 5.96%

제목: Airlines face price hikes, lower margins as Iran war pressures business
예측: 🟡 NEUTRAL
  negative: 3.06%
  neutral:  83.45%
  positive: 13.49%

제목: ‘There’s No Other Choice’: War-Hit Asian Buyers Grab Russian Oil
예측: 🟡 NEUTRAL
  negative: 5.66%
  neutral:  86.38%
  positive: 7.95%

제목: Stock market today: Dow soars 1,000 points, S&P 500 and Nasdaq surge in upbeat end to brutal quarter
예측: 🟢 POSITIVE
  negative: 2.37%
  neutral:  23.56%
  positive: 74.06%

제목: Oil plunges, stocks soar after Iranian pr

In [67]:
print(f"{'제목':<45} {'본문전체':^25} {'앞3문장':^25}")
print("="*95)
for i in range(len(titles)):
    title = titles[i][:40]
    result_a, pred_a = get_prediction(full_texts[i])
    result_c, pred_c = get_prediction(summary_texts[i])
    
    a_str = f"{result_a}({pred_a[np.argmax(pred_a)]:.0%})"
    c_str = f"{result_c}({pred_c[np.argmax(pred_c)]:.0%})"
    
    print(f"{title:<45} {a_str:^25} {c_str:^25}")


제목                                                      본문전체                      앞3문장           
‘There’s No Other Choice’: War-Hit Asian            neutral(89%)              neutral(86%)       
Stock market today: Dow soars 1,000 poin            positive(74%)             positive(74%)      
Oil plunges, stocks soar after Iranian p            neutral(89%)              neutral(89%)       
Airlines face price hikes, lower margins            neutral(56%)              neutral(83%)       
